好的，如果说“数学计算”是 **Hello World** 级别的案例，那么下面我为你展示一个**真正体现 Semantic Kernel (SK) 核心威力**的案例。

这个案例的核心在于：**原生代码（Native）与语义代码（Semantic）的无缝混合编排**。

### 案例场景：AI 自动化投研分析师 (The AI Investment Researcher)

**为什么要用 SK 做这个？**
在实际业务中，我们经常遇到这种需求：
1.  **硬数据获取**：去数据库或 API 查股价、查财报（这是 Python 的强项，LLM 做不了）。
2.  **软信息总结**：去网上读新闻、读推特，分析市场情绪（这是 LLM 的强项，Python 做不了）。
3.  **决策输出**：结合 1 和 2，写一份投资建议书。

在 SK 中，我们可以把这三个步骤封装在一个 **Plugin** 类里，让 AI 像调用普通函数一样“调用”一个 Prompt。

---

### 完整代码实现

这个案例将展示两个“特别”的点：
1.  **混合 Plugin**：一个类里既有 Python 代码，又有 Prompt 模板。
2.  **嵌套调用**：AI 为了完成任务，会先调 Python 拿数据，再调 Prompt 写分析。

```python
import asyncio
import os
import dotenv
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion, OpenAIChatPromptExecutionSettings
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior
from semantic_kernel.contents import ChatHistory
from semantic_kernel.functions import kernel_function

dotenv.load_dotenv()

# --- 定义一个混合型 Plugin ---
class InvestmentPlugin:
    """
    一个包含 硬数据获取(Native) 和 软逻辑分析(Semantic) 的混合插件
    """

    # 【能力 1】Native Function: 获取实时股价 (模拟 API)
    @kernel_function(description="获取股票的当前价格", name="get_stock_price")
    def get_stock_price(self, ticker: str) -> str:
        print(f"    [系统日志] 正在调用 API 查询 {ticker} 价格...")
        # 模拟数据
        data = {
            "AAPL": "175.50 USD",
            "TSLA": "210.20 USD",
            "NVDA": "800.00 USD"
        }
        return data.get(ticker.upper(), "未知价格")

    # 【能力 2】Native Function: 获取近期新闻 (模拟 爬虫)
    @kernel_function(description="获取股票的最近重大新闻", name="get_market_news")
    def get_market_news(self, ticker: str) -> str:
        print(f"    [系统日志] 正在搜索 {ticker} 的新闻...")
        # 模拟数据
        news = {
            "AAPL": "苹果发布了新的 Vision Pro，市场反应平平。",
            "TSLA": "特斯拉 Cybertruck 产能爬坡顺利，但自动驾驶面临监管调查。",
            "NVDA": "英伟达发布最新 AI 芯片，算力提升 300%。"
        }
        return news.get(ticker.upper(), "无重大新闻")

    # 【能力 3】Semantic Function: 情感分析 (Prompt)
    # 这是一个特别之处：我们在 Python 代码里定义了一个 Prompt 作为一个功能函数
    # 这样 AI 就可以像调用 get_price 一样调用 analyze_sentiment
    @kernel_function(description="根据新闻内容分析市场情绪 (Positive/Negative/Neutral)", name="analyze_sentiment")
    async def analyze_sentiment(self, news_text: str, kernel: Kernel) -> str:
        print(f"    [系统日志] 正在调用 LLM 分析新闻情绪...")

        # 这是一个 "函数内的函数"
        # 我们动态创建一个 Prompt Function
        analyze_prompt = """
        请分析以下新闻的市场情绪：
        {{$input}}

        只输出以下单词之一：Positive, Negative, Neutral
        """

        # 动态创建并执行这个 Prompt
        func = kernel.create_function_from_prompt(analyze_prompt)
        result = await kernel.invoke(func, input=news_text)
        return str(result)

async def main():
    # 1. 初始化
    kernel = Kernel()
    service_id = "default"
    kernel.add_service(
        OpenAIChatCompletion(
            service_id=service_id,
            ai_model_id="gpt-4o-mini",
            api_key=os.getenv("OPENAI_API_KEY"),
        )
    )

    # 2. 注册我们的混合插件
    kernel.add_plugin(InvestmentPlugin(), plugin_name="InvestTools")

    # 3. 开启自动工具调用
    settings = OpenAIChatPromptExecutionSettings(
        service_id=service_id,
        function_choice_behavior=FunctionChoiceBehavior.Auto()
    )

    # 4. 获取 Chat 服务
    chat_service = kernel.get_service(service_id)
    history = ChatHistory()

    # --- 这是一个非常复杂的指令，需要 AI 自动拆解 ---
    user_query = "帮我分析一下 Tesla (TSLA) 的情况，我需要知道现在的价格，以及基于最近的新闻，我现在应该买入还是卖出？"

    print(f"[用户指令]: {user_query}")
    history.add_user_message(user_query)

    # 5. 执行
    # AI 将会自动执行以下思维链：
    # 1. 调用 get_stock_price("TSLA") -> 拿到价格
    # 2. 调用 get_market_news("TSLA") -> 拿到新闻
    # 3. (关键点) AI 可能会自己决定调用 analyze_sentiment(新闻) -> 拿到情绪
    # 4. 最终结合所有信息，生成建议

    result = await chat_service.get_chat_message_contents(
        chat_history=history,
        settings=settings,
        kernel=kernel
    )

    print("\n" + "="*50)
    print(f"[最终分析报告]:\n{result[0]}")
    print("="*50)

if __name__ == "__main__":
    import asyncio
    asyncio.run(main())
```

---

### 这个案例“特别”在哪里？

#### 1. "Prompt as Code" (Prompt 即代码)
请注意看 `analyze_sentiment` 这个函数。
*   它对 AI 来说，就是一个普通的函数，输入字符串，输出字符串。
*   但它的**内部实现**不是 Python 的 `if/else`，而是**调用了另一个 LLM Prompt**。
*   **威力**：这意味着你可以用简单的 Python 函数签名，封装极度复杂的认知任务。调用者（或顶层 Agent）根本不需要知道里面是跑的代码还是跑的模型。

#### 2. 多步推理与依赖链 (Dependency Chaining)
当你运行这段代码时，你会发现后台日志不仅是简单的调用。
AI 为了回答“买入还是卖出”，它必须：
1.  先拿数据（Native）。
2.  再拿新闻（Native）。
3.  **把新闻传给情绪分析函数**（Native -> Semantic）。
4.  最后汇总。

SK 的 `FunctionChoiceBehavior.Auto()` 能够处理这种参数传递，即**上一个工具的输出，作为下一个工具的输入**。

### 还有更高级的玩法吗？

如果你觉得这还不够特别，SK 还有一种 **"Planner" (规划器)** 模式（虽然现在逐渐被 Auto Tool Call 取代，但思想依然存在），它可以实现：

**场景：智能家居的“意图理解”**

*   **Plugin A (Light)**: `turn_on`, `turn_off`, `set_color`
*   **Plugin B (Music)**: `play_song`, `set_volume`
*   **User Says**: "我要在这个昏暗的房间里读会儿书，来点轻松的背景音。"

**SK 的处理方式**：
它不会直接匹配关键词，而是利用 **Semantic Function** 理解“读书”需要的环境：
1.  语义理解：读书 -> 需要明亮的光线 -> 调用 `Light.set_brightness(80)`。
2.  语义理解：轻松背景音 -> 调用 `Music.play("Classical/Lo-Fi")`。

这种**从模糊意图到精确函数参数的映射**，正是 Semantic Kernel "Skills" 最迷人的地方。